# Lesson 1 — Prices are fixed point, and float breaks them where it matters

Prices arrive as decimal strings on a grid that can be as fine as a hundredth of a cent, and every decision here is a comparison at the fourth decimal place. The engine parses to Decimal and never converts.

**The rule.** `0.1 + 0.2 ≠ 0.3 in binary64; eight legs at 0.1250 must total exactly 1.0000`

**When it holds.** Wherever a price is compared to another price or to a dollar — which on this tab is everywhere.

**When it fails.** A float engine is not slightly wrong. It is right on every case a casual test would try and wrong on the marginal ones, which are the only cases an arbitrage engine looks at.

| | |
|---|---|
| Lesson id | `fixedpoint` |
| Pane it appears on | `books` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/money.py`, `modules/coherence/kernel/grid.py` |
| Tests that go red if it stops being true | `tests/test_coherence_money.py`, `tests/test_coherence_grid.py`, `tests/test_coherence_no_float.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. The sign of `total - 1` is the whole answer

In [ ]:
from modules.coherence.kernel.money import (
    CENTICENT,
    MoneyError,
    ceil_to_centicent,
    contracts,
    floor_to_precision,
    format_dollars,
    one_minus,
    parse_dollars,
    parse_fp,
)

legs = ["0.7000", "0.2000", "0.1000"]

# Accumulated leg by leg, which is what pricing a basket looks like.
exact = Decimal(0)
running = 0.0
for leg in legs:
    exact += parse_dollars(leg)
    running += float(leg)

print(f"  three legs of a mutually exclusive family: {legs}")
print(f"    Decimal, added leg by leg : {exact}  ->  total - 1 = {exact - 1}")
print(f"    float,   added leg by leg : {running!r}  ->  total - 1 = {running - 1.0!r}")
print()
print(f"    Decimal says this basket is a Dutch book: {exact < 1}")
print(f"    float   says this basket is a Dutch book: {running < 1.0}")
print()
print("  The sign of total - 1 is the entire answer, and the float engine has just")
print("  invented a Dutch book out of three exact prices.")
print()
print("  Worse, it is not reproducible. CPython's own sum() compensates, so the SAME legs")
print(f"  come to {sum(float(leg) for leg in legs)!r} there — the fault depends on how the loop happened to")
print("  be written, which is the hardest kind of defect to find in a running system.")

## 2. Ticks that are exact, and floats that are not

In [ ]:
print(f"  0.1 + 0.2 == 0.3 in binary64 : {0.1 + 0.2 == 0.3}")
print(f"  the same sum in Decimal      : {parse_dollars('0.1') + parse_dollars('0.2') == parse_dollars('0.3')}")
print()
eight = [parse_dollars("0.1250")] * 8
ten = [parse_dollars("0.1000")] * 10
print(f"  eight legs at 0.1250 total {sum(eight, Decimal(0))} exactly")
print(f"  ten legs at 0.1000 total   {sum(ten, Decimal(0))} exactly")

running_ten = 0.0
for _ in range(10):
    running_ten += 0.1
print(f"  ten float legs at 0.1, added one at a time, total {running_ten!r}")

## 3. A price we cannot read is refused, never defaulted

In [ ]:
# There is no sensible fallback for an unparseable price: zero is a legal Kalshi
# price, so any guess invents liquidity. The parser refuses and the caller
# reports the market as unreadable.
for candidate in (0.42, 42, True, "", "0.4200000", "cheap"):
    try:
        parsed = parse_dollars(candidate)
    except MoneyError as exc:
        print(f"  {candidate!r:>12}  refused: {exc}")
    else:
        print(f"  {candidate!r:>12}  accepted as {parsed}")

## 4. The exchange's own quanta

In [ ]:
hundredths = parse_fp("0.09")
print(f"  parse_fp('0.09') = {hundredths} hundredths = {contracts(hundredths)} contracts")
print(f"  fractional contracts are unconditional here, at {contracts(1)} granularity")
print()
raw = Decimal("0.07") * contracts(hundredths) * Decimal("0.3301") * one_minus(Decimal("0.3301"))
fee = ceil_to_centicent(raw)
print(f"  a raw trade fee of {raw}")
print(f"  ceils UP to {fee}, because the quantum is {CENTICENT}")
change = -Decimal("0.3301") * contracts(hundredths) - fee
floored = floor_to_precision(change, Decimal("0.01"))
print(f"  the balance change {change} floors to {floored}")
print(f"  and the shortfall {change - floored} is charged as the rounding fee")
print()
print(f"  a YES bid at 0.4200 is a NO ask at {format_dollars(one_minus(parse_dollars('0.4200')))}")